# S5 Demo — Bias & Explainability with SageMaker Clarify (Topic 5.1)
We audit **our own** Session 2 churn model, facet = `SeniorCitizen` (a seeded disparity exists —
the point is that the audit **catches** it). Clarify jobs take ~20 min: **always pre-run**
(day before), demo from the saved report in `fallback_reports/`. SHAP runs locally either way.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

## The Clarify job config (pre-run — never live)
```python
from sagemaker import clarify
processor = clarify.SageMakerClarifyProcessor(role=role, instance_count=1,
                                              instance_type='ml.m5.large', sagemaker_session=sess)
bias_config = clarify.BiasConfig(label_values_or_threshold=[1],
                                 facet_name='SeniorCitizen', facet_values_or_threshold=[1])
# processor.run_bias(data_config, bias_config, model_config)   # ~20 min → fallback_reports/
```

## Pre-training bias — on the raw data, no model needed (live, local)

In [ ]:
from course_utils import data
import pandas as pd
df = data.load_processed('telco_clean')
g = df.groupby('SeniorCitizen')['Churn'].agg(['mean', 'count'])
g['share'] = g['count'] / len(df)
print(g.round(3))
dpl = g.loc[1, 'mean'] - g.loc[0, 'mean']
print(f'\nDPL (difference in positive proportions of labels): {dpl:.3f}')
print('class imbalance of the facet itself:', round(g.loc[0, 'share'] - g.loc[1, 'share'], 3))

**Read it in sentences:** senior citizens are a small share of customers AND churn at a visibly higher rate → any model trained on this inherits the pattern. The Clarify report flags exactly these two numbers (CI, DPL) plus post-training metrics (disparate impact, accuracy difference) from the saved job.

## Post-training: does the MODEL treat the groups differently?

In [ ]:
from course_utils import models
mdl = models.load_model('churn_xgb_v1')
X, y = data.prep_features(df)
prob = mdl.predict_proba(X)[:, 1]
pred = (prob >= 0.5).astype(int)
aud = pd.DataFrame({'senior': df['SeniorCitizen'], 'y': y, 'pred': pred})
rates = aud.groupby('senior')[['pred', 'y']].mean().round(3)
print(rates)
print('disparate impact (pred rate ratio non-senior/senior):',
      round(rates.loc[0, 'pred'] / rates.loc[1, 'pred'], 3))

## SHAP — why does the model say what it says? (local, live)
XGBoost computes SHAP values natively (`pred_contribs`) — no extra library needed.
(The `shap` package adds fancier plots but is version-sensitive with XGBoost; Clarify
uses the same SHAP math server-side.)

In [ ]:
import numpy as np, xgboost as xgb
dm = xgb.DMatrix(X.iloc[:500])
sv = mdl.get_booster().predict(dm, pred_contribs=True)[:, :-1]   # last col = bias term
global_imp = pd.Series(np.abs(sv).mean(axis=0), index=X.columns).nlargest(10)
global_imp.sort_values().plot.barh(title='Global feature attribution (mean |SHAP|)', figsize=(7, 4));

In [ ]:
i = 7   # one customer, one explanation — what you tell the retention agent
contrib = pd.Series(sv[i], index=X.columns).sort_values(key=abs, ascending=False)[:6]
print(f'customer {i}: churn prob {prob[i]:.2f}')
print(contrib.round(3))

**Wrap:** report caught the seeded imbalance → mitigation options (reweighting, per-group thresholds, data collection) each carry a trade-off → that judgment call is the student task.